# Modelo Final — Recomendador Híbrido *popularity-first* (NeuMF + prior)

Este é o **capstone** do sistema. Ele consolida a abordagem final:

1. **Offline:** NeuMF treinado com objetivo de ranking (classificação binária de
   interação com amostragem de negativos);
2. **Prior:** popularidade contada no treino (o sinal mais forte nesta base);
3. **Combinação:** `score = z(log1p(pop)) + alpha * z(NeuMF)`, com `alpha` escolhido
   na **validação** por política conservadora;
4. **Avaliação final:** protocolo *leave-last-out* contra baselines honestos.

> A persistência dos artefatos e o registro no MLflow são feitos pelo estágio
> `treinamento_modelo` do pipeline (`src/scripts/otimizacao_modelo.py`). Aqui
> reproduzimos o treino para **diagnóstico e narrativa**, sem sobrescrever os
> artefatos versionados pelo DVC.

## 1. Configuração do ambiente

In [1]:
import sys
from pathlib import Path

# Localiza a raiz do projeto (onde está o pyproject.toml) para importar `src.*`.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

from src.config.model_utils import (
    TOP_K,
    RankingCheckpoint,
    build_item_catalog,
    build_training_pairs,
    categories_for_products,
    compute_popularity_prior,
    create_ranking_model,
    evaluate_hybrid,
    evaluate_most_popular,
    evaluate_random_baseline,
    evaluate_ranking,
    recommend_products,
    select_alpha,
    to_model_inputs,
    train_ranking_model,
    train_validation_test_split,
)
from src.config.paths import DADOS_TRATADOS

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("Projeto:", ROOT)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

Projeto: C:\Users\flavi\Documents\GitHub\Sistema_Recomendacao_Arquitetura-MLOps
TensorFlow: 2.19.0 | Keras: 3.9.0


## 2. Análise da base

In [2]:
dados = pd.read_parquet(DADOS_TRATADOS)
interacoes_por_cliente = dados.groupby("Customer Name")["Product ID"].nunique()

print("Interações:", dados.shape)
print("Clientes:", dados["Customer Name"].nunique(), "| Produtos:", dados["Product ID"].nunique())
print(
    f"Itens distintos por cliente -> média {interacoes_por_cliente.mean():.2f} | "
    f"mediana {interacoes_por_cliente.median():.0f} | máx {interacoes_por_cliente.max():d}"
)
densidade = 100 * dados[["Customer Name", "Product ID"]].drop_duplicates().shape[0] / (
    dados["Customer Name"].nunique() * dados["Product ID"].nunique()
)
print(f"Densidade da matriz cliente x produto: {densidade:.4f}%")

Interações: (9800, 18)
Clientes: 793 | Produtos: 1861
Itens distintos por cliente -> média 12.30 | mediana 11 | máx 34
Densidade da matriz cliente x produto: 0.6607%


## 3. Split *leave-last-out* e pares de treino

O **último** item de cada cliente vai para o teste, o **penúltimo** para a validação e
o restante para o treino. O treino recebe negativos amostrados (produtos não
consumidos).

In [3]:
encoders = {
    "customer": LabelEncoder().fit(dados["Customer Name"]),
    "product": LabelEncoder().fit(dados["Product ID"]),
    "category": LabelEncoder().fit(dados["Category"]),
    "subcategory": LabelEncoder().fit(dados["Sub-Category"]),
}
num_clientes = len(encoders["customer"].classes_)
num_produtos = len(encoders["product"].classes_)
num_categorias = len(encoders["category"].classes_)
num_subcategorias = len(encoders["subcategory"].classes_)

catalogo = build_item_catalog(
    dados, encoders["product"], encoders["category"], encoders["subcategory"]
)
train_pairs, val_pairs, test_pairs = train_validation_test_split(
    dados, encoders["customer"], encoders["product"]
)
train_customers, train_products, train_labels = build_training_pairs(
    train_pairs, num_produtos, n_negatives=8, seed=42
)
train_categorias, train_subcategorias = categories_for_products(train_products, catalogo)

print(f"Pares -> treino: {len(train_pairs)} | validação: {len(val_pairs)} | teste: {len(test_pairs)}")
print("Amostras de treino (positivos+negativos):", len(train_labels))

Pares -> treino: 8171 | validação: 776 | teste: 776
Amostras de treino (positivos+negativos): 73539


## 4. Treino do NeuMF com early stopping por NDCG@7

O `RankingCheckpoint` avalia o **NDCG@7 de validação** a cada época e restaura os
pesos da melhor época — monitorar a métrica-alvo evita escolher um modelo que separa
bem positivos/negativos (AUC alta) mas ordena mal o catálogo.

In [4]:
modelo = create_ranking_model(
    num_customers=num_clientes,
    num_products=num_produtos,
    num_categories=num_categorias,
    num_subcategories=num_subcategorias,
)
checkpoint = RankingCheckpoint(val_pairs, train_pairs, catalogo, k=TOP_K, patience=5, verbose=1)

train_ranking_model(
    modelo,
    train_customers=train_customers,
    train_products=train_products,
    categories=train_categorias,
    subcategories=train_subcategorias,
    labels=train_labels,
    callbacks=[checkpoint],
    epochs=40,
    batch_size=256,
    verbose=2,
)

print("Melhor época:", checkpoint.best_epoch, "| melhor NDCG@7 de validação:", round(checkpoint.best, 5))

Epoch 1/40


    [ranking] época 1: val_ndcg@7=0.00356 (melhor=0.00356 na época 1)
288/288 - 3s - 10ms/step - auc: 0.5090 - loss: 0.4411 - val_ndcg@7: 0.0036


Epoch 2/40


    [ranking] época 2: val_ndcg@7=0.00221 (melhor=0.00356 na época 1)
288/288 - 2s - 5ms/step - auc: 0.5319 - loss: 0.3594 - val_ndcg@7: 0.0022


Epoch 3/40


    [ranking] época 3: val_ndcg@7=0.00139 (melhor=0.00356 na época 1)
288/288 - 2s - 5ms/step - auc: 0.5284 - loss: 0.3549 - val_ndcg@7: 0.0014


... (15 outputs left out)

    [ranking] época 11: val_ndcg@7=0.00357 (melhor=0.00440 na época 7)
288/288 - 2s - 5ms/step - auc: 0.5396 - loss: 0.3491 - val_ndcg@7: 0.0036


Epoch 12/40


    [ranking] época 12: val_ndcg@7=0.00137 (melhor=0.00440 na época 7)
288/288 - 2s - 5ms/step - auc: 0.5368 - loss: 0.3491 - val_ndcg@7: 0.0014


Melhor época: 7 | melhor NDCG@7 de validação: 0.0044


## 5. Seleção de `alpha` (híbrido *popularity-first*)

O prior de popularidade é o sinal-base. O `alpha` controla quanto do sinal
personalizado entra no score. A política é **conservadora**: só adota `alpha > 0` se
ele superar o puramente popular (`alpha=0`) por uma margem mínima na validação.

In [5]:
prior_popularidade = compute_popularity_prior(train_pairs, num_produtos)
alpha, historico_alpha = select_alpha(
    modelo, val_pairs, train_pairs, catalogo, prior_popularidade,
    alphas=(0.0, 0.1, 0.25, 0.5, 1.0), k=TOP_K,
)

print(f"NDCG@{TOP_K} de validação por alpha:")
for a, v in historico_alpha.items():
    print(f"  alpha={a:<5} -> {v:.5f}")
print(f"alpha escolhido: {alpha}")

NDCG@7 de validação por alpha:
  alpha=0.0   -> 0.00478
  alpha=0.1   -> 0.00337
  alpha=0.25  -> 0.00212
  alpha=0.5   -> 0.00105
  alpha=1.0   -> 0.00214
alpha escolhido: 0.0


## 6. Avaliação final no teste + baselines

In [6]:
metricas = {
    "Híbrido (final)": evaluate_hybrid(
        modelo, test_pairs, train_pairs, catalogo, prior_popularidade, alpha, k=TOP_K
    ),
    "NeuMF puro": evaluate_ranking(modelo, test_pairs, train_pairs, catalogo, k=TOP_K),
    "Mais populares": evaluate_most_popular(test_pairs, train_pairs, num_produtos, k=TOP_K),
    "Aleatório (esperança)": evaluate_random_baseline(test_pairs, train_pairs, num_produtos, k=TOP_K),
}

linhas = []
for nome, m in metricas.items():
    linhas.append({
        "Modelo": nome,
        f"NDCG@{TOP_K}": m.get(f"ndcg@{TOP_K}", np.nan),
        f"MAP@{TOP_K}": m.get(f"map@{TOP_K}", np.nan),
        f"HitRate@{TOP_K}": m.get(f"hit_rate@{TOP_K}", np.nan),
        f"Precision@{TOP_K}": m.get(f"precision@{TOP_K}", np.nan),
        f"Recall@{TOP_K}": m.get(f"recall@{TOP_K}", np.nan),
        f"F1@{TOP_K}": m.get(f"f1@{TOP_K}", np.nan),
    })

tabela = pd.DataFrame(linhas).set_index("Modelo").round(4).fillna("—")
tabela

,NDCG@7,MAP@7,HitRate@7,Precision@7,Recall@7,F1@7
Modelo,,,,,,
Híbrido (final),0.0027,0.002,0.0052,0.0007,0.0052,0.0013
NeuMF puro,0.0009,0.0004,0.0026,0.0004,0.0026,0.0006
Mais populares,0.0027,0.002,0.0052,0.0007,0.0052,0.0013
Aleatório (esperança),—,—,0.0038,0.0005,0.0038,—


Os traços (**—**) marcam métricas sem forma fechada para o baseline **aleatório**
(reportamos apenas a esperança **analítica exata** de Hit-Rate, Precision e Recall,
em vez de aproximar NDCG/MAP/F1 por simulação).

## 7. Registro no MLflow (tracking + modelo registrado)

O tracking é **não-bloqueante**: se o servidor estiver indisponível, o pipeline
prossegue. Usamos o backend local SQLite por padrão (ou um servidor DagsHub, se
`RECOMENDACAO_MLFLOW_TRACKING_URI` estiver definido).

In [7]:
from src.config.mlflow_tracking import log_model_metrics, setup_mlflow

setup_mlflow()

metricas_log = {
    f"ndcg_{TOP_K}": metricas["Híbrido (final)"][f"ndcg@{TOP_K}"],
    f"map_{TOP_K}": metricas["Híbrido (final)"][f"map@{TOP_K}"],
    f"hit_rate_{TOP_K}": metricas["Híbrido (final)"][f"hit_rate@{TOP_K}"],
    f"ndcg_neumf_puro_{TOP_K}": metricas["NeuMF puro"][f"ndcg@{TOP_K}"],
    f"ndcg_baseline_populares_{TOP_K}": metricas["Mais populares"][f"ndcg@{TOP_K}"],
    "alpha": float(alpha),
}

X_exemplo = to_model_inputs(
    test_pairs["c"].to_numpy()[:200], test_pairs["p"].to_numpy()[:200], catalogo
)

run_id = log_model_metrics(
    modelo, X_exemplo, None, metricas_log, model_name="recomendador_neumf"
)
print("Run ID:", run_id)

MLflow configurado com URI: sqlite:///C:/Users/flavi/Documents/GitHub/Sistema_Recomendacao_Arquitetura-MLOps/mlflow.db | Experimento: recomendacao_produtos


2026/11/18 12:25:19 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


2026/11/18 12:25:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Registered model 'recomendador_neumf' already exists. Creating a new version of this model...
Created version '4' of model 'recomendador_neumf'.


Modelo registrado com sucesso. Run ID: eea48df78aa64e82ab83bbc647e400eb
Run ID: eea48df78aa64e82ab83bbc647e400eb


## 8. Recomendações de exemplo

In [8]:
CLIENTE = "Irene Maddox"
recomendacoes = recommend_products(
    modelo, CLIENTE, dados, encoders["customer"], encoders["product"], catalogo,
    top_k=TOP_K, popularity=prior_popularidade, alpha=alpha,
)
recomendacoes

,ranking,Product ID,Product Name,Category,Sub-Category,score
0,1,OFF-PA-10001970,Xerox 1881,Office Supplies,Paper,2.667958
1,2,TEC-AC-10003832,Logitech P710e Mobile Speakerphone,Technology,Accessories,2.667958
2,3,OFF-BI-10004140,Avery Non-Stick Binders,Office Supplies,Binders,2.253192
3,4,TEC-AC-10003628,Logitech 910-002974 M325 Wireless Mouse for We...,Technology,Accessories,2.253192
4,5,OFF-BI-10000977,Ibico Plastic Spiral Binding Combs,Office Supplies,Binders,2.253192
5,6,FUR-CH-10002647,"Situations Contoured Folding Chairs, 4/Set",Furniture,Chairs,2.094879
6,7,TEC-AC-10002049,Logitech G19 Programmable Gaming Keyboard,Technology,Accessories,2.094879


## 9. Conclusão

- O sistema final é um **híbrido popularity-first** com métricas de **ranking**
  honestas e baselines explícitos.
- A escolha de `alpha` é **data-driven** (validação) e conservadora: nesta base, o
  prior de popularidade venceu (`alpha=0`), o que é uma conclusão legítima e
  auditável — não um resultado forçado.
- Os artefatos de produção (modelo, encoders e `hybrid_config_*.pkl`) são versionados
  pelo DVC no estágio `treinamento_modelo`; o notebook 03 demonstra o consumo deles.

Trabalhos futuros: *ranking* por par (BPR/lista), features contextuais (sazonalidade),
e avaliação com *bootstrap* para intervalos de confiança das métricas.